# Data Processing - Female Representation in Critically Acclaimed Games

This notebook covers the whole **collection, integration, cleaning and preparation** stage of the study. It produces two CSV files, exported at the end:

- `data/processed/games.csv` - one record per game (game level)
- `data/processed/characters.csv` - one record per character (character level)

Both files are read by `02_data_analysis.ipynb`.

**Execution and reproducibility note**

The RAWG collection retrieves **all** games released between 2010-01-01 and 2022-12-31 with a Metacritic score between 80 and 100 (filter `metacritic=80,100`), ordered by Metacritic score in descending order (`ordering=-metacritic`), paging through the API until the last page. The date, time and parameters of each collection are saved automatically to `data/raw/rawg_collection_info.json`. The collection used in the study was run on 28 September 2026 at 20:33 (UTC-3) and returned 1,280 titles.

Because the RAWG database is continuously updated (scores, ratings and platforms), a new collection may return different results.

To reproduce the exact sample of the study, keep `download_datasets = False` (the default). The notebook then reads the snapshots stored in `data/raw/`, and no API key or Kaggle account is needed.

In [1]:
import json
import os
import re
import time
from unicodedata import normalize

import numpy as np
import pandas as pd
import requests

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 150)

RAW_DIR = os.path.join("..", "data", "raw")
PROCESSED_DIR = os.path.join("..", "data", "processed")

In [2]:
# False (default): read the raw snapshots stored in data/raw/ and reproduce the study sample exactly.
# True: download fresh data from the RAWG API and from Kaggle (requires RAWG_API_KEY in ../.env
#       and Kaggle credentials) and overwrite the files in data/raw/.

download_datasets = False

## 1. Data collection

### 1.1 RAWG API

All titles released between 2010-01-01 and 2022-12-31 with a Metacritic score between 80 and 100, ordered by Metacritic score (`-metacritic`), 40 games per page (the API maximum) until the last page. The API key is read from the `.env` file at the repository root (not versioned; see `.env.example`).

In [3]:
def load_env(path=os.path.join("..", ".env")):
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    k, v = line.split("=", 1)
                    os.environ.setdefault(k.strip(), v.strip())

load_env()
RAWG_API_KEY = os.getenv("RAWG_API_KEY")
BASE_URL = "https://api.rawg.io/api/games"

# Collection parameters: all games from 2010 to 2022 with a Metacritic score between 80 and 100,
# ordered by Metacritic score (descending). page_size = 40 is the API maximum.
COLLECTION_PARAMS = {
    "dates": "2010-01-01,2022-12-31",
    "metacritic": "80,100",
    "ordering": "-metacritic",
    "page_size": 40,
}

def fetch_rawg_games(max_pages=500, sleep=1):
    rows = []
    page = 1
    total_api = None
    while page <= max_pages:
        params = {"key": RAWG_API_KEY, "page": page, **COLLECTION_PARAMS}
        r = requests.get(BASE_URL, params=params)

        if r.status_code != 200:
            print("API error:", r.status_code, "on page", page)
            break

        data = r.json()
        total_api = data.get("count", total_api)

        for game in data["results"]:
            rows.append({
                "game_id": game.get("id"),
                "name": game.get("name"),
                "released": game.get("released"),
                "rating": game.get("rating"),
                "ratings_count": game.get("ratings_count"),
                "metacritic": game.get("metacritic"),
                "platforms": game.get("platforms"),
                "genres": game.get("genres"),
                "developers": game.get("developers"),
                "publishers": game.get("publishers")
            })

        print(f"Page {page} collected ({len(rows)} of {total_api} titles)")

        # stop when the API reports no next page
        if not data.get("next"):
            break
        page += 1
        time.sleep(sleep)

    return pd.DataFrame(rows), total_api

if download_datasets:
    started = pd.Timestamp.now()
    df_rawg, total_api = fetch_rawg_games()

    print("RAWG dataset size:", df_rawg.shape, "| total reported by the API:", total_api)
    if total_api is not None and len(df_rawg) < total_api:
        print("WARNING: the collection returned fewer titles than the total reported by the API.")

    os.makedirs(RAW_DIR, exist_ok=True)
    df_rawg.to_csv(os.path.join(RAW_DIR, "rawg_games_raw.csv"), index=False)

    # collection log, for documentation in the paper
    info = {
        "collected_at": started.strftime("%Y-%m-%d %H:%M"),
        "endpoint": BASE_URL,
        "parameters": COLLECTION_PARAMS,
        "titles_collected": int(len(df_rawg)),
        "total_reported_by_api": total_api,
    }
    with open(os.path.join(RAW_DIR, "rawg_collection_info.json"), "w") as f:
        json.dump(info, f, ensure_ascii=False, indent=2)
    print(info)

### 1.2 Kaggle dataset - *Gender Representation in Video Games* (Palomar, 2022)

Three tables: games, characters and the sexualization index of each character.

In [4]:
if download_datasets:
    import kagglehub

    dataset_path = kagglehub.dataset_download("br33sa/gender-representation-in-video-games")

    df_games = pd.read_csv(os.path.join(dataset_path, "games.grivg.csv"))
    df_characters = pd.read_csv(os.path.join(dataset_path, "characters.grivg.csv"))
    df_sexualization = pd.read_csv(os.path.join(dataset_path, "sexualization.grivg.csv"))

    # Save the Kaggle snapshot next to the RAWG snapshot
    os.makedirs(RAW_DIR, exist_ok=True)
    df_games.to_csv(os.path.join(RAW_DIR, "kaggle_games.csv"), index=False)
    df_characters.to_csv(os.path.join(RAW_DIR, "kaggle_characters.csv"), index=False)
    df_sexualization.to_csv(os.path.join(RAW_DIR, "kaggle_sexualization.csv"), index=False)

### 1.3 Loading the stored snapshots (default)

In [5]:
if not download_datasets:
    df_rawg = pd.read_csv(os.path.join(RAW_DIR, "rawg_games_raw.csv"))
    df_games = pd.read_csv(os.path.join(RAW_DIR, "kaggle_games.csv"))
    df_characters = pd.read_csv(os.path.join(RAW_DIR, "kaggle_characters.csv"))
    df_sexualization = pd.read_csv(os.path.join(RAW_DIR, "kaggle_sexualization.csv"))

print("RAWG:", df_rawg.shape)
with open(os.path.join(RAW_DIR, "rawg_collection_info.json")) as f:
    print("RAWG collection:", json.load(f))
print("Games:", df_games.shape)
print("Characters:", df_characters.shape)
print("Sexualization:", df_sexualization.shape)

RAWG: (1280, 10)
RAWG collection: {'collected_at': '2026-09-28 20:33', 'endpoint': 'https://api.rawg.io/api/games', 'parameters': {'dates': '2010-01-01,2022-12-31', 'metacritic': '80,100', 'ordering': '-metacritic', 'page_size': 40}, 'titles_collected': 1280, 'total_reported_by_api': 1280}
Games: (64, 28)
Characters: (637, 12)
Sexualization: (637, 6)


In [6]:
df_games.head()

,Game_Id,Title,Release,Series,Genre,Sub-genre,Developer,Publisher,Country,Platform,PEGI,Customizable_main,Protagonist,Protagonist_Non_Male,Relevant_males,Relevant_no_males,Percentage_non_male,Criteria,Director,Total_team,female_team,Team_percentage,Metacritic,Destructoid,IGN,GameSpot,Avg_Reviews,Unnamed: 27
0,GTAV,Grand Theft Auto V,Nov-13,GTA,Action-adventure,Action-adventure,Rockstar North,Rockstar Games,GBR,Multi,18,No,3,0,9,2,18%,MS,M,7,0,0%,9.7,9.0,10.0,9.0,9.4,NaN
1,PSS,Pokémon Sword/Shield,Nov-19,Pokémon,RPG,RPG,Game Freak,Nintendo,JPN,Nintendo Switch,7,Yes,1,1,4,5,56%,MS,M,9,1,11%,8.0,7.0,9.3,9.0,8.3,NaN
2,CODMW,Call of Duty: Modern Warfare,Oct-19,Call of Duty,Action,FPS,Infinity Ward,Activision,USA,Multi,18,No,3,1,5,1,17%,MS,M,11,0,0%,8.0,8.0,8.0,7.0,7.8,NaN
3,RDR2,Red Dead Redemption 2,Dec-18,Red Dead,Action-adventure,Action-adventure,Rockstar Studios,Rockstar Games,USA,Multi,18,No,1,0,17,7,29%,SR,M,7,0,0%,9.7,9.5,10.0,9.0,9.6,NaN
4,SMO,Super Mario Odyssey,Oct-17,Super Mario,Action-adventure,Action-adventure,Nintendo EDP,Nintendo,JPN,Nintendo Switch,7,No,1,0,3,2,40%,SR,M,11,1,9%,9.7,9.5,10.0,10.0,9.8,NaN


In [7]:
df_characters.head()

,Name,Gender,Game,Age,Age_range,Playable,Sexualization,Id,Species,Side,Relevance,Romantic_Interest
0,Farah,Female,CODMW,27,Adult,1,0,CODMW_Farah,Human,P,PA,No
1,Protagonist,Custom,PSS,Teenager,Teenager,1,0,PSS_Protagonist,Human,P,PA,No
2,Magnolia,Female,PSS,Elderly,Elderly,0,0,PSS_Magnolia,Human,P,SC,No
3,Sonia,Female,PSS,26,Adult,0,0,PSS_Sonia,Human,P,SC,No
4,Marnie,Female,PSS,Teenager,Teenager,0,0,PSS_Marnie,Human,B,MC,No


In [8]:
df_sexualization.head()

,Id,Sexualized_clothing,Trophy,Damsel in Distress,Sexualized Cutscenes,Total
0,CODMW_Farah,0,0,0,0,0
1,PSS_Protagonist,0,0,0,0,0
2,PSS_Magnolia,0,0,0,0,0
3,PSS_Sonia,0,0,0,0,0
4,PSS_Marnie,0,0,0,0,0


In [9]:
print("Games columns")
print(df_games.columns)

print("\nCharacters columns")
print(df_characters.columns)

print("\nSexualization columns")
print(df_sexualization.columns)

Games columns
Index(['Game_Id', 'Title', 'Release', 'Series', 'Genre', 'Sub-genre', 'Developer', 'Publisher', 'Country', 'Platform', 'PEGI', 'Customizable_main',
       'Protagonist', 'Protagonist_Non_Male', 'Relevant_males', 'Relevant_no_males', 'Percentage_non_male', 'Criteria', 'Director', 'Total_team',
       'female_team', 'Team_percentage', 'Metacritic ', 'Destructoid', 'IGN', 'GameSpot', 'Avg_Reviews', 'Unnamed: 27'],
      dtype='object')

Characters columns
Index(['Name', 'Gender', 'Game', 'Age', 'Age_range', 'Playable', 'Sexualization', 'Id', 'Species', 'Side', 'Relevance', 'Romantic_Interest'], dtype='object')

Sexualization columns
Index(['Id', 'Sexualized_clothing', 'Trophy', 'Damsel in Distress', 'Sexualized Cutscenes', 'Total'], dtype='object')


## 2. RAWG preparation and selection criterion

In [10]:
import ast

def extract_platform_names(platform_list):
    # When read back from CSV, the RAWG list columns are stored as strings
    if isinstance(platform_list, str):
        try:
            platform_list = ast.literal_eval(platform_list)
        except (ValueError, SyntaxError):
            return []
    try:
        return [p["platform"]["name"] for p in platform_list]
    except (TypeError, KeyError):
        return []

df_rawg["platform_names"] = df_rawg["platforms"].apply(extract_platform_names)
df_rawg["year"] = pd.to_datetime(df_rawg["released"], errors="coerce").dt.year
df_rawg["year"].value_counts().sort_index()

year
2010    100
2011     97
2012     97
2013    108
2014    100
2015    128
2016    137
2017    134
2018     95
2019    103
2020     90
2021     62
2022     29
Name: count, dtype: int64

In [11]:
df_rawg

,game_id,name,released,rating,ratings_count,metacritic,platforms,genres,developers,publishers,platform_names,year
0,27036,Super Mario Galaxy 2,2010-05-23,4.33,427,97,"[{'platform': {'id': 7, 'name': 'Nintendo Swit...","[{'id': 83, 'name': 'Platformer', 'slug': 'pla...",NaN,NaN,"[Nintendo Switch, Wii U, Wii]",2010
1,28026,Super Mario Odyssey,2017-10-27,4.42,1944,97,"[{'platform': {'id': 7, 'name': 'Nintendo Swit...","[{'id': 83, 'name': 'Platformer', 'slug': 'pla...",NaN,NaN,[Nintendo Switch],2017
2,22511,The Legend of Zelda: Breath of the Wild,2017-03-03,4.47,3225,97,"[{'platform': {'id': 7, 'name': 'Nintendo Swit...","[{'id': 3, 'name': 'Adventure', 'slug': 'adven...",NaN,NaN,"[Nintendo Switch, Wii U]",2017
3,401808,Half-Life 2: Update,2015-03-27,4.13,127,96,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'id': 2, 'name': 'Shooter', 'slug': 'shooter'}]",NaN,NaN,[PC],2015
4,28,Red Dead Redemption 2,2018-10-26,4.59,5575,96,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'id': 4, 'name': 'Action', 'slug': 'action'}]",NaN,NaN,"[PC, Xbox One, PlayStation 4]",2018
...,...,...,...,...,...,...,...,...,...,...,...,...
1275,452635,Kena: Bridge of Spirits,2021-09-21,3.97,469,80,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'id': 2, 'name': 'Shooter', 'slug': 'shooter...",NaN,NaN,"[PC, PlayStation 5, PlayStation 4, Xbox Series...",2021
1276,10612,CRYPTARK,2015-10-07,3.18,21,80,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'id': 2, 'name': 'Shooter', 'slug': 'shooter...",NaN,NaN,"[PC, PlayStation 4, macOS, Linux]",2015
1277,2390,Plague Inc: Evolved,2015-09-18,3.98,673,80,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'id': 40, 'name': 'Casual', 'slug': 'casual'...",NaN,NaN,"[PC, Xbox One, PlayStation 4, Nintendo Switch,...",2015
1278,4466,Catherine,2011-02-17,4.14,513,80,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'id': 3, 'name': 'Adventure', 'slug': 'adven...",NaN,NaN,"[PC, Xbox One, Xbox 360, PlayStation 3]",2011


In [12]:
# Selection criterion: Metacritic score of 80 or higher
# (already applied by the API filter; kept here to document the criterion)
df_rawg["AAA"] = np.where(df_rawg["metacritic"] >= 80, 1, 0)
df_rawg["AAA"].value_counts()

AAA
1    1280
Name: count, dtype: int64

In [13]:
df_rawg_aaa = df_rawg[df_rawg["AAA"] == 1].copy()
df_rawg_aaa["name"]

0                          Super Mario Galaxy 2
1                           Super Mario Odyssey
2       The Legend of Zelda: Breath of the Wild
3                           Half-Life 2: Update
4                         Red Dead Redemption 2
                         ...                   
1275                    Kena: Bridge of Spirits
1276                                   CRYPTARK
1277                        Plague Inc: Evolved
1278                                  Catherine
1279         Lego Star Wars: The Skywalker Saga
Name: name, Length: 1280, dtype: object

## 3. Integration of RAWG and Kaggle

Titles are matched after normalization: lower case, apostrophes and accents removed, years in parentheses removed, punctuation replaced by spaces, common edition suffixes stripped and isolated Roman numerals converted to Arabic numerals. Titles whose names differ between the sources are matched through manually verified equivalences.

In [14]:
# Isolated Roman numerals (II to XV) are converted to Arabic numerals, so that
# "Dark Souls II" and "Dark Souls 2" (or "Original Sin II" and "Original Sin 2")
# have the same normalized title. An isolated "I" is not converted, so that
# titles such as "I Was a Teenage Exocolonist" are not altered.
ROMAN = {
    "ii": "2", "iii": "3", "iv": "4", "v": "5", "vi": "6", "vii": "7",
    "viii": "8", "ix": "9", "x": "10", "xi": "11", "xii": "12",
    "xiii": "13", "xiv": "14", "xv": "15",
}

def clean_title(title):
    title = title.lower()

    # remove apostrophes (straight and typographic) before general punctuation,
    # so that "Thief's" and "Thief’s" both become "thiefs"
    title = re.sub(r"['’‘`´]", "", title)

    title = normalize("NFKD", title).encode("ascii", "ignore").decode("utf-8")

    # remove a year in parentheses, e.g. "God of War (2018)"
    title = re.sub(r"\(\s*(19|20)\d{2}\s*\)", " ", title)

    # remove punctuation
    title = re.sub(r"[^a-z0-9\s]", " ", title)

    # remove common edition words
    remove_words = [
        "game of the year edition",
        "definitive edition",
        "complete edition",
        "remastered",
        "ultimate edition"
    ]
    for w in remove_words:
        title = title.replace(w, "")

    # isolated Roman numerals -> Arabic numerals
    title = " ".join(ROMAN.get(tok, tok) for tok in title.split())

    title = re.sub(r"\s+", " ", title).strip()
    return title

In [15]:
df_rawg_aaa["title_clean"] = df_rawg_aaa["name"].apply(clean_title)
df_games["title_clean"] = df_games["Title"].apply(clean_title)

# Manual equivalences: Kaggle titles that correspond to a RAWG title with a
# different subtitle or name, checked manually (same game, same original
# release). Each entry points to the exact RAWG name.
manual_aliases = {
    "DQXI": "DRAGON QUEST XI: Echoes of an Elusive Age",   # Dragon Quest XI (2017)
    "TWD":  "The Walking Dead: Season 1",                  # The Walking Dead (Telltale, 2012)
    "SK":   "Shovel Knight",                               # "Shovel of Hope" is the later name of the original 2014 campaign
    "PSS":  "Pokémon Sword and Shield",                    # Pokémon Sword/Shield (2019)
    "CACW": "Chicory: A Colorful Tale",                    # correct title of the 2021 game; "Colorful World" in Kaggle
}
for game_id, rawg_name in manual_aliases.items():
    df_games.loc[df_games["Game_Id"] == game_id, "title_clean"] = clean_title(rawg_name)

df_rawg_aaa

,game_id,name,released,rating,ratings_count,metacritic,platforms,genres,developers,publishers,platform_names,year,AAA,title_clean
0,27036,Super Mario Galaxy 2,2010-05-23,4.33,427,97,"[{'platform': {'id': 7, 'name': 'Nintendo Swit...","[{'id': 83, 'name': 'Platformer', 'slug': 'pla...",NaN,NaN,"[Nintendo Switch, Wii U, Wii]",2010,1,super mario galaxy 2
1,28026,Super Mario Odyssey,2017-10-27,4.42,1944,97,"[{'platform': {'id': 7, 'name': 'Nintendo Swit...","[{'id': 83, 'name': 'Platformer', 'slug': 'pla...",NaN,NaN,[Nintendo Switch],2017,1,super mario odyssey
2,22511,The Legend of Zelda: Breath of the Wild,2017-03-03,4.47,3225,97,"[{'platform': {'id': 7, 'name': 'Nintendo Swit...","[{'id': 3, 'name': 'Adventure', 'slug': 'adven...",NaN,NaN,"[Nintendo Switch, Wii U]",2017,1,the legend of zelda breath of the wild
3,401808,Half-Life 2: Update,2015-03-27,4.13,127,96,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'id': 2, 'name': 'Shooter', 'slug': 'shooter'}]",NaN,NaN,[PC],2015,1,half life 2 update
4,28,Red Dead Redemption 2,2018-10-26,4.59,5575,96,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'id': 4, 'name': 'Action', 'slug': 'action'}]",NaN,NaN,"[PC, Xbox One, PlayStation 4]",2018,1,red dead redemption 2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1275,452635,Kena: Bridge of Spirits,2021-09-21,3.97,469,80,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'id': 2, 'name': 'Shooter', 'slug': 'shooter...",NaN,NaN,"[PC, PlayStation 5, PlayStation 4, Xbox Series...",2021,1,kena bridge of spirits
1276,10612,CRYPTARK,2015-10-07,3.18,21,80,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'id': 2, 'name': 'Shooter', 'slug': 'shooter...",NaN,NaN,"[PC, PlayStation 4, macOS, Linux]",2015,1,cryptark
1277,2390,Plague Inc: Evolved,2015-09-18,3.98,673,80,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'id': 40, 'name': 'Casual', 'slug': 'casual'...",NaN,NaN,"[PC, Xbox One, PlayStation 4, Nintendo Switch,...",2015,1,plague inc evolved
1278,4466,Catherine,2011-02-17,4.14,513,80,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'id': 3, 'name': 'Adventure', 'slug': 'adven...",NaN,NaN,"[PC, Xbox One, Xbox 360, PlayStation 3]",2011,1,catherine


In [16]:
rawg_set = set(df_rawg_aaa["title_clean"])
kaggle_set = set(df_games["title_clean"])

intersection = rawg_set.intersection(kaggle_set)
print("Possible matches:", len(intersection))

# Kaggle games left out of the sample, with the reason,
# for manual review of possible false negatives in the matching
df_rawg["title_clean"] = df_rawg["name"].fillna("").apply(clean_title)
left_out = df_games[~df_games["title_clean"].isin(intersection)]
print(f"\nKaggle games left out of the sample: {len(left_out)}")
for _, g in left_out.iterrows():
    in_rawg = df_rawg[df_rawg["title_clean"] == g["title_clean"]]
    if in_rawg.empty:
        reason = "not in the RAWG collection (Metacritic < 80 or missing on RAWG)"
    else:
        reason = f"in RAWG, but Metacritic = {in_rawg['metacritic'].tolist()} (< 80 or missing)"
    print(f"  {g['Game_Id']:7s} {g['Title'][:40]:40s} -> {reason}")

Possible matches: 60

Kaggle games left out of the sample: 4
  CODBO2  Call of Duty: Black Ops 2                -> not in the RAWG collection (Metacritic < 80 or missing on RAWG)
  IWATE   I Was a Teenage Exocolonist              -> not in the RAWG collection (Metacritic < 80 or missing on RAWG)
  IMM     Immortality                              -> not in the RAWG collection (Metacritic < 80 or missing on RAWG)
  COL     Child of Light                           -> not in the RAWG collection (Metacritic < 80 or missing on RAWG)


In [17]:
df_merged = df_rawg_aaa.merge(
    df_games,
    on="title_clean",
    how="inner"
)

# Pairs whose original names differ between the sources, for manual checking
different = df_merged[df_merged["name"].str.lower().str.strip() != df_merged["Title"].str.lower().str.strip()]
print(f"Pairs with different names in RAWG and Kaggle (to check): {len(different)}")
print(different[["Game_Id", "Title", "name", "released", "metacritic"]].to_string(index=False))
df_merged

Pairs with different names in RAWG and Kaggle (to check): 19
Game_Id                              Title                                          name   released  metacritic
   DOS2           Divinity:Original Sin II                      Divinity: Original Sin 2 2017-09-14          95
   TLOU                     The Last of Us                     The Last Of Us Remastered 2014-07-29          95
   DOS2           Divinity:Original Sin II Divinity: Original Sin 2 - Definitive Edition 2018-08-31          94
    GOW                         God of War                             God of War (2018) 2018-04-20          94
    HLA                     Half-Life:Alyx                               Half-Life: Alyx 2020-03-23          93
    UC4         Uncharted 4: A Thief's End                    Uncharted 4: A Thief’s End 2016-05-10          93
    TWD                   The Walking Dead                    The Walking Dead: Season 1 2012-04-23          92
    TW3           The Witcher 3: Wild Hunt 

,game_id,name,released,rating,ratings_count,metacritic,platforms,genres,developers,publishers,platform_names,year,AAA,title_clean,Game_Id,Title,Release,Series,Genre,Sub-genre,Developer,Publisher,Country,Platform,PEGI,Customizable_main,Protagonist,Protagonist_Non_Male,Relevant_males,Relevant_no_males,Percentage_non_male,Criteria,Director,Total_team,female_team,Team_percentage,Metacritic,Destructoid,IGN,GameSpot,Avg_Reviews,Unnamed: 27
0,28026,Super Mario Odyssey,2017-10-27,4.42,1944,97,"[{'platform': {'id': 7, 'name': 'Nintendo Swit...","[{'id': 83, 'name': 'Platformer', 'slug': 'pla...",NaN,NaN,[Nintendo Switch],2017,1,super mario odyssey,SMO,Super Mario Odyssey,Oct-17,Super Mario,Action-adventure,Action-adventure,Nintendo EDP,Nintendo,JPN,Nintendo Switch,7,No,1,0,3,2,40%,SR,M,11,1,9%,9.7,9.5,10.0,10.0,9.8,NaN
1,22511,The Legend of Zelda: Breath of the Wild,2017-03-03,4.47,3225,97,"[{'platform': {'id': 7, 'name': 'Nintendo Swit...","[{'id': 3, 'name': 'Adventure', 'slug': 'adven...",NaN,NaN,"[Nintendo Switch, Wii U]",2017,1,the legend of zelda breath of the wild,BOTW,The Legend of Zelda: Breath of the Wild,Mar-17,The Legend of Zelda,Action-adventure,Action-adventure,Nintendo EDP,Nintendo,JPN,Multi,12,No,1,0,13,5,28%,SR,M,8,0,0%,9.7,10.0,10.0,10.0,9.9,NaN
2,28,Red Dead Redemption 2,2018-10-26,4.59,5575,96,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'id': 4, 'name': 'Action', 'slug': 'action'}]",NaN,NaN,"[PC, Xbox One, PlayStation 4]",2018,1,red dead redemption 2,RDR2,Red Dead Redemption 2,Dec-18,Red Dead,Action-adventure,Action-adventure,Rockstar Studios,Rockstar Games,USA,Multi,18,No,1,0,17,7,29%,SR,M,7,0,0%,9.7,9.5,10.0,9.0,9.6,NaN
3,10073,Divinity: Original Sin 2,2017-09-14,4.38,1690,95,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'id': 10, 'name': 'Strategy', 'slug': 'strat...",NaN,NaN,"[PC, Xbox One, PlayStation 4, Nintendo Switch]",2017,1,divinity original sin 2,DOS2,Divinity:Original Sin II,Sep-17,Divinity,RPG,RPG,Larian Studios,Larian Studios,BEL,Multi,18,Yes,1,1,7,4,36%,TR,M,7,1,14%,9.4,9.0,9.6,10.0,9.5,NaN
4,3636,The Last Of Us Remastered,2014-07-29,4.69,2859,95,"[{'platform': {'id': 18, 'name': 'PlayStation ...","[{'id': 3, 'name': 'Adventure', 'slug': 'adven...",NaN,NaN,[PlayStation 4],2014,1,the last of us,TLOU,The Last of Us,Jun-13,NaN,Action-adventure,Action-adventure,Naughty Dog,Sony Computer Entertainment,USA,PS3,18,No,1,0,6,3,33%,TR,M,7,0,0%,9.5,10.0,10.0,8.0,9.4,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60,1358,"Papers, Please",2013-08-08,4.37,1786,85,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'id': 34, 'name': 'Educational', 'slug': 'ed...",NaN,NaN,"[PC, iOS, Android, macOS, Linux, PS Vita]",2013,1,papers please,PP,"Papers, Please",Aug-13,NaN,Simulation,Puzzle Simulation,3909 LLC,3909 LLC,USA,Multi,16,No,1,0,4,2,33%,TR,M,1,0,0%,8.9,NaN,8.7,8.0,8.5,NaN
61,3350,Broken Age,2014-01-28,3.65,689,81,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'id': 3, 'name': 'Adventure', 'slug': 'adven...",NaN,NaN,"[PC, Xbox One, PlayStation 4, Nintendo Switch,...",2014,1,broken age,BRA,Broken Age,Jan-14,NaN,Adventure,Point-and-click adventure,Double Fine Productions,Double Fine Productions,USA,Computer,12,No,2,1,9,7,44%,TR,M,6,1,17%,7.9,9.5,9.5,7.0,8.5,NaN
62,282825,Pokémon Sword and Shield,2019-11-15,3.72,492,80,"[{'platform': {'id': 7, 'name': 'Nintendo Swit...","[{'id': 3, 'name': 'Adventure', 'slug': 'adven...",NaN,NaN,[Nintendo Switch],2019,1,pokemon sword and shield,PSS,Pokémon Sword/Shield,Nov-19,Pokémon,RPG,RPG,Game Freak,Nintendo,JPN,Nintendo Switch,7,Yes,1,1,4,5,56%,MS,M,9,1,11%,8.0,7.0,9.3,9.0,8.3,NaN
63,323065,Call of Duty: Modern Warfare (2019),2019-10-25,3.78,741,80,"[{'platform': {'id': 4, 'name': 'PC', 'slug': ...","[{'id': 2, 'name': 'Shooter', 'slug': 'shooter...",NaN,NaN,"[PC, Xbox One, PlayStation 4]",2019,1,call of duty modern warfare

In [18]:
df_aaa = df_merged.copy()

## 4. Missing data and sample traceability

In [19]:
# ==============================================================
# MISSING DATA - df_aaa (game level)
# Identifies which core and control variables have missing values
# ==============================================================

# Core variables for the hypotheses and control variables
game_vars = [
    "year", "Genre", "metacritic", "rating",
    "Director", "Country", "Team_percentage", "Percentage_non_male",
    "Metacritic ", "IGN", "GameSpot", "Destructoid", "Avg_Reviews"
]

missing_game = (
    df_aaa[game_vars]
    .isnull()
    .sum()
    .reset_index()
)
missing_game.columns = ["Variable", "N missing"]
missing_game["% missing"] = (missing_game["N missing"] / len(df_aaa) * 100).round(2)
missing_game["Type"] = [
    "Control", "Control", "Selection criterion", "RAWG audience",
    "Team", "Team", "Team", "Representation",
    "Critics", "Critics", "Critics", "Critics", "Critics"
]
missing_game = missing_game.sort_values("% missing", ascending=False)

print("=== Missing data - df_aaa (games) ===\n")
print(f"Total rows in the sample: {len(df_aaa)}")
missing_game

=== Missing data - df_aaa (games) ===

Total rows in the sample: 65


,Variable,N missing,% missing,Type
11,Destructoid,5,7.69,Critics
9,IGN,1,1.54,Critics
10,GameSpot,1,1.54,Critics
0,year,0,0.00,Control
1,Genre,0,0.00,Control
2,metacritic,0,0.00,Selection criterion
3,rating,0,0.00,RAWG audience
4,Director,0,0.00,Team
5,Country,0,0.00,Team
6,Team_percentage,0,0.00,Team


In [20]:
# ==============================================================
# SAMPLE TRACEABILITY
# ==============================================================

n_collected = len(df_rawg)
n_aaa       = int(df_rawg["AAA"].sum())
n_match     = len(df_merged)
n_final     = df_aaa["Game_Id"].nunique()

steps = {
    "Step": [
        "Collection via RAWG API (2010-2022, Metacritic 80-100)",
        "Selection criterion: Metacritic >= 80",
        "Match with Kaggle dataset (inner merge)",
        "Final sample (unique games)"
    ],
    "N games": [n_collected, n_aaa, n_match, n_final],
    "Excluded": ["-", n_collected - n_aaa, n_aaa - n_match, n_match - n_final],
    "Reason for exclusion": [
        "-",
        "Metacritic score < 80 or missing",
        "Title without a match between RAWG and Kaggle",
        "Duplicates removed (editions and remasters of the same title)"
    ]
}

df_traceability = pd.DataFrame(steps)
print("=== Sample traceability ===\n")
df_traceability

=== Sample traceability ===



,Step,N games,Excluded,Reason for exclusion
0,"Collection via RAWG API (2010-2022, Metacritic...",1280,-,-
1,Selection criterion: Metacritic >= 80,1280,0,Metacritic score < 80 or missing
2,Match with Kaggle dataset (inner merge),65,1215,Title without a match between RAWG and Kaggle
3,Final sample (unique games),60,5,Duplicates removed (editions and remasters of ...


The table above documents the sample losses at each step of corpus construction: collection of titles through the RAWG API (2010-2022), application of the selection criterion (Metacritic score of 80 or higher), matching with the Kaggle dataset by normalized title, and consolidation into a single entry per game of the duplicates produced by special editions and remastered versions. Because the API query already applies the `metacritic=80,100` filter, the selection step excludes no titles in this collection (1,280 -> 1,280); it is kept in the table to document the criterion. Of the 64 games in the Kaggle dataset, 60 enter the sample; the other four (Call of Duty: Black Ops 2, Child of Light, I Was a Teenage Exocolonist and Immortality) are not in the collection, either because they have no Metacritic score on RAWG or because their score is below 80.

In [21]:
aaa_game_ids = df_aaa["Game_Id"]
df_characters_aaa = df_characters[
    df_characters["Game"].isin(aaa_game_ids)
].copy()

print("Characters in the selected games:", df_characters_aaa.shape)
df_characters_aaa

Characters in the selected games: (578, 12)


,Name,Gender,Game,Age,Age_range,Playable,Sexualization,Id,Species,Side,Relevance,Romantic_Interest
0,Farah,Female,CODMW,27,Adult,1,0,CODMW_Farah,Human,P,PA,No
1,Protagonist,Custom,PSS,Teenager,Teenager,1,0,PSS_Protagonist,Human,P,PA,No
2,Magnolia,Female,PSS,Elderly,Elderly,0,0,PSS_Magnolia,Human,P,SC,No
3,Sonia,Female,PSS,26,Adult,0,0,PSS_Sonia,Human,P,SC,No
4,Marnie,Female,PSS,Teenager,Teenager,0,0,PSS_Marnie,Human,B,MC,No
...,...,...,...,...,...,...,...,...,...,...,...,...
632,Beni,Male,KBOS,Infant,Infant,0,0,KBOS_Beni,Human,P,SC,No
633,Taro,Male,KBOS,Child,Child,0,0,KBOS_Taro,Human,P,SC,No
634,Rusu,Male,KBOS,Adult,Adult,0,0,KBOS_Rusu,Human,P,SC,No
635,Adira,Female,KBOS,Adult,Adult,0,0,KBOS_Adira,Human,P,SC,No


In [22]:
df_sex_full = df_characters_aaa.merge(
    df_sexualization[["Id", "Sexualized_clothing", "Trophy",
                      "Damsel in Distress", "Sexualized Cutscenes", "Total"]],
    on="Id",
    how="left"
)

print("=== Composite sexualization index - descriptive statistics ===\n")
print(df_sex_full.groupby("Gender")["Total"].describe().round(2).to_string())

print("\n=== Share of characters with each dimension, by gender ===")
dims = ["Sexualized_clothing", "Trophy", "Damsel in Distress", "Sexualized Cutscenes"]
for dim in dims:
    print(f"\n{dim}:")
    print(
        df_sex_full.groupby("Gender")[dim]
        .mean()
        .mul(100)
        .round(2)
        .rename("% with dimension present")
        .to_string()
    )

=== Composite sexualization index - descriptive statistics ===

            count  mean   std  min  25%  50%  75%  max
Gender                                                
Custom        7.0  0.00  0.00  0.0  0.0  0.0  0.0  0.0
Female      183.0  0.26  0.66  0.0  0.0  0.0  0.0  3.0
Male        359.0  0.02  0.15  0.0  0.0  0.0  0.0  2.0
Non-binary   29.0  0.03  0.19  0.0  0.0  0.0  0.0  1.0

=== Share of characters with each dimension, by gender ===

Sexualized_clothing:
Gender
Custom        0.00
Female        9.29
Male          0.56
Non-binary    0.00

Trophy:
Gender
Custom        0.00
Female        3.28
Male          0.28
Non-binary    0.00

Damsel in Distress:
Gender
Custom        0.00
Female        8.20
Male          0.56
Non-binary    3.45

Sexualized Cutscenes:
Gender
Custom        0.00
Female        5.46
Male          0.28
Non-binary    0.00


In [23]:
df_aaa_unique = df_aaa.drop_duplicates(subset="Game_Id")

table1 = (
    df_aaa_unique
    .groupby(["year", "Genre"])
    .size()
    .unstack(fill_value=0)
)

# Row and column totals
table1["Total"] = table1.sum(axis=1)
table1.loc["Total"] = table1.sum()

print("Sample distribution by year and game genre")
print(f"\nUnique games in the sample: {df_aaa_unique['Game_Id'].nunique()}")
print(f"Characters (all): {len(df_characters_aaa)}")
print(f"Playable characters: {(df_characters_aaa['Playable'] == 1).sum()}")
print(f"Playable female characters: {((df_characters_aaa['Playable'] == 1) & (df_characters_aaa['Gender'] == 'Female')).sum()}")
print(f"Playable male characters: {((df_characters_aaa['Playable'] == 1) & (df_characters_aaa['Gender'] == 'Male')).sum()}")
table1

Sample distribution by year and game genre

Unique games in the sample: 60
Characters (all): 578
Playable characters: 111
Playable female characters: 29
Playable male characters: 65


Genre,Action,Action-adventure,Adventure,Interactive Story,RPG,Simulation,Total
year,,,,,,,
2012,3,0,2,0,0,0,5
2013,2,1,0,0,0,1,4
2014,1,2,1,0,2,0,6
2015,1,2,0,0,2,0,5
2016,2,2,0,0,3,0,7
2017,1,3,2,0,3,0,9
2018,1,3,0,1,0,0,5
2019,0,2,0,0,3,0,5
2020,1,3,0,1,1,0,6


The final integrated sample holds information at two levels of analysis. At the game level, `df_aaa` contains 60 unique titles with structural data from the RAWG API (release year, Metacritic score, platforms) enriched with variables from the Kaggle dataset, including the composition of the development team, the director's gender, scores from specialized outlets and the representation indicators computed by the authors of the original dataset. At the character level, `df_characters_aaa` contains 578 characters across the 60 games, of which 111 are playable: 65 male (58.56%) and 29 female (26.13%). The formal hypotheses H1 and H3 are tested on this playable subset, since the unit of analysis defined in the methodology is the playable female character, while H2 and H4 also use the full set of characters. Action-adventure is the most frequent genre (21 titles), followed by RPG (15) and Action (14). Note that the year distribution in the table above is computed before the deduplication of Section 5, so for some games it still reflects the release year of a later edition kept by the merge; in the final sample (see `02_data_analysis.ipynb`), titles concentrate in 2017 (9 games), followed by 2015, 2019 and 2020 (6 games each).

### 4.1 Missing data before and after cleaning

In [24]:
# ==============================================================
# BEFORE x AFTER CLEANING
# Shows the impact of data cleaning on df_aaa and df_characters_aaa
# before any change to the original dataframes
# ==============================================================

core_game_vars = [
    "Game_Id", "Title", "year", "Genre", "metacritic", "rating",
    "Director", "Country", "Team_percentage", "Percentage_non_male",
    "Metacritic ", "IGN", "GameSpot", "Destructoid", "Avg_Reviews",
    "Protagonist_Non_Male"
]

missing_aaa = (
    df_aaa[core_game_vars]
    .isnull()
    .sum()
    .reset_index()
)
missing_aaa.columns = ["Variable", "N missing"]
missing_aaa["% missing"] = (missing_aaa["N missing"] / len(df_aaa) * 100).round(2)
missing_aaa = missing_aaa[missing_aaa["N missing"] > 0].sort_values("% missing", ascending=False)

print("=== Missing data - df_aaa ===")
print(f"Total rows: {len(df_aaa)}\n")
missing_aaa

=== Missing data - df_aaa ===
Total rows: 65



,Variable,N missing,% missing
13,Destructoid,5,7.69
11,IGN,1,1.54
12,GameSpot,1,1.54


In [25]:
core_char_vars = [
    "Name", "Gender", "Game", "Age", "Age_range",
    "Playable", "Sexualization", "Species", "Side",
    "Relevance", "Romantic_Interest"
]

missing_chars = (
    df_characters_aaa[core_char_vars]
    .isnull()
    .sum()
    .reset_index()
)
missing_chars.columns = ["Variable", "N missing"]
missing_chars["% missing"] = (missing_chars["N missing"] / len(df_characters_aaa) * 100).round(2)
missing_chars = missing_chars[missing_chars["N missing"] > 0].sort_values("% missing", ascending=False)

print("=== Missing data - df_characters_aaa ===")
print(f"Total rows: {len(df_characters_aaa)}\n")
missing_chars

=== Missing data - df_characters_aaa ===
Total rows: 578



,Variable,N missing,% missing


In [26]:
# Simulated cleaning: drop rows missing variables used by the hypotheses
df_aaa_sim = df_aaa.drop_duplicates(subset="Game_Id").dropna(subset=["year", "Genre", "metacritic"])
df_chars_sim = df_characters_aaa.dropna(subset=["Gender", "Relevance", "Playable"])

def playable_counts(df):
    playable = df["Playable"] == 1
    fem = (playable & (df["Gender"] == "Female")).sum()
    male = (playable & (df["Gender"] == "Male")).sum()
    return playable.sum(), fem, male, f"{fem / max(playable.sum(), 1) * 100:.2f}%"

pre, post = playable_counts(df_characters_aaa), playable_counts(df_chars_sim)

comp = pd.DataFrame({
    "Metric": [
        "Games (df_aaa rows)",
        "Unique games (Game_Id)",
        "Characters (df_characters_aaa rows)",
        "Playable characters",
        "Playable female characters",
        "Playable male characters",
        "% female among playable"
    ],
    "Before cleaning": [len(df_aaa), df_aaa["Game_Id"].nunique(), len(df_characters_aaa), *pre],
    "After cleaning (simulation)": [len(df_aaa_sim), df_aaa_sim["Game_Id"].nunique(), len(df_chars_sim), *post],
})

print("=== Before x after cleaning ===\n")
print(comp.to_string(index=False))

=== Before x after cleaning ===

                             Metric Before cleaning After cleaning (simulation)
                Games (df_aaa rows)              65                          60
             Unique games (Game_Id)              60                          60
Characters (df_characters_aaa rows)             578                         578
                Playable characters             111                         111
         Playable female characters              29                          29
           Playable male characters              65                          65
            % female among playable          26.13%                      26.13%


The missing-data analysis shows that the impact of cleaning is limited and does not affect the hypotheses. At the game level, the core variables `year`, `Genre` and `metacritic` have no missing values; the only operation needed is deduplication by `Game_Id`, which reduces `df_aaa` from 65 rows to 60 unique titles - the difference comes from duplicated rows created by the RAWG x Kaggle merge for games with several editions. The only fields with missing values (Destructoid, IGN and GameSpot) are critic scores used in the exploratory analyses; correlations with them are computed on the games with a valid value in each variable, without excluding games from the sample. At the character level, `df_characters_aaa` has no missing values in any core variable, so the 111 playable characters - 29 female (26.13%) and 65 male - remain fully available for the hypothesis tests, with no imputation or further exclusion.

## 5. Final cleaning and export

### 5.1 Deduplication and release-year checks

The merge between RAWG and Kaggle produces more than one row for games with several editions on RAWG (65 -> 60 games). For each game, the edition whose RAWG release year matches the `Release` year in the Kaggle dataset is kept; ties are broken by keeping the earliest edition (original release). This prevents, for example, *Call of Duty: Modern Warfare* (2019) from being replaced by *Modern Warfare Remastered* (2016), a remaster of a different game.

In [27]:
kaggle_year = df_aaa["Release"].str.extract(r"-(\d{2})$")[0].astype(float) + 2000
df_aaa["_same_year"] = (df_aaa["year"] == kaggle_year).astype(int)
df_aaa = (df_aaa.sort_values(["_same_year", "released"], ascending=[False, True])
          .drop_duplicates(subset="Game_Id")
          .drop(columns="_same_year")
          .reset_index(drop=True))
print(f"df_aaa after deduplication: {len(df_aaa)} unique games")

df_aaa after deduplication: 60 unique games


In [28]:
# Release-year checks for cases that are not edition duplicates in the merge,
# verified against an external source (Wikipedia, 16 September 2026).
# The year comes from RAWG; these entries document the typos in the Kaggle Release field.
manual_corrections = {
    "FC3": {
        "correct_year": 2012,
        "source": "Far Cry 3 was released in Nov/Dec 2012; the Kaggle Release value "
                  "'Nov-19' is a typo and should be 'Nov-12'."
    },
    "HA": {
        "correct_year": 2020,
        "source": "Hades left early access with its 1.0 release on 2020-09-17; the Kaggle "
                  "Release value 'Sep-21' is incorrect and should be 'Sep-20'."
    },
}

for game_id, info in manual_corrections.items():
    df_aaa.loc[df_aaa["Game_Id"] == game_id, "year"] = info["correct_year"]

print("Release-year checks (year taken from RAWG):")
for game_id, info in manual_corrections.items():
    print(f"  {game_id}: year = {info['correct_year']}")

Release-year checks (year taken from RAWG):
  FC3: year = 2012
  HA: year = 2020


In [29]:
# Check on the edition actually kept in df_aaa:
# the RAWG release year should match the year in the Kaggle Release field
df_merged_checked = df_merged[
    df_merged.set_index(["Game_Id", "released"]).index.isin(
        df_aaa.set_index(["Game_Id", "released"]).index)
].copy()

def kaggle_release_year(release_str):
    match = re.search(r"-(\d{2})$", str(release_str))
    if match:
        year = int(match.group(1))
        return 2000 + year if year < 50 else 1900 + year
    return None

df_merged_checked["kaggle_year"] = df_merged_checked["Release"].apply(kaggle_release_year)
df_merged_checked["rawg_year"] = pd.to_datetime(df_merged_checked["released"]).dt.year

still_divergent = df_merged_checked[
    df_merged_checked["kaggle_year"] != df_merged_checked["rawg_year"]
][["Game_Id", "Title", "Release", "released"]]

print(f"Games still divergent after the correction: {len(still_divergent)}")
still_divergent

Games still divergent after the correction: 2


,Game_Id,Title,Release,released
11,HA,Hades,Sep-21,2020-09-17
37,FC3,Far Cry 3,Nov-19,2012-11-28


The two remaining divergences (FC3 and HA) are typos in the Kaggle `Release` field; the `year` variable used in the analysis comes from the RAWG release date, which is correct for both games.

### 5.2 Conversion of percentage columns (% of women in the team and % of non-male characters)

In [30]:
df_aaa["team_pct_num"] = (
    df_aaa["Team_percentage"]
    .str.replace("%", "", regex=False)
    .astype(float)
    .round(2)
)

df_aaa["char_pct_num"] = (
    df_aaa["Percentage_non_male"]
    .str.replace("%", "", regex=False)
    .astype(float)
    .round(2)
)

print("Converted columns: team_pct_num and char_pct_num")
df_aaa[["Title", "team_pct_num", "char_pct_num"]].head(10)

Converted columns: team_pct_num and char_pct_num


,Title,team_pct_num,char_pct_num
0,Journey,14.0,100.0
1,The Walking Dead,0.0,39.0
2,Borderlands 2,0.0,33.0
3,Hotline Miami,0.0,20.0
4,BioShock Infinite,0.0,57.0
5,The Swapper,0.0,67.0
6,The Last of Us,0.0,33.0
7,"Papers, Please",0.0,33.0
8,Grand Theft Auto V,0.0,18.0
9,Broken Age,17.0,44.0


### 5.3 Building `games.csv` (game level)

Redundant columns inherited from the RAWG x Kaggle merge are dropped (`Unnamed: 27`, `title_clean`, `AAA` - the selection flag, always 1 after the filter - and the RAWG fields `game_id`/`name`/`released`/`genres`, already covered by the Kaggle fields `Game_Id`/`Title`/`Release`/`Genre`), as well as the RAWG list columns not used in any analysis (`platforms`, `developers`, `publishers`, `platform_names`).

The two Metacritic fields (RAWG and Kaggle) and the RAWG audience fields are renamed to make the source of each score explicit, since the merge produced two almost identical names (`metacritic` and `'Metacritic '`, with a trailing space).

In [31]:
games = df_aaa.rename(columns={
    "Metacritic ":   "Metacritic_Kaggle",
    "metacritic":    "Metacritic_RAWG",
    "rating":        "Rating_RAWG_Audience",
    "ratings_count": "RatingsCount_RAWG",
}).copy()

games_columns = [
    "Game_Id", "Title", "Release", "year", "Genre", "Sub-genre", "Series",
    "Developer", "Publisher", "Country", "Platform", "PEGI", "Customizable_main",
    "Protagonist", "Protagonist_Non_Male", "Relevant_males", "Relevant_no_males",
    "Percentage_non_male", "char_pct_num", "Criteria", "Director", "Total_team",
    "female_team", "Team_percentage", "team_pct_num",
    "Metacritic_Kaggle", "Metacritic_RAWG", "Rating_RAWG_Audience", "RatingsCount_RAWG",
    "Destructoid", "IGN", "GameSpot", "Avg_Reviews",
]

games = games[games_columns]

print(f"games.csv: {games.shape[0]} rows x {games.shape[1]} columns")
games.head()

games.csv: 60 rows x 33 columns


,Game_Id,Title,Release,year,Genre,Sub-genre,Series,Developer,Publisher,Country,Platform,PEGI,Customizable_main,Protagonist,Protagonist_Non_Male,Relevant_males,Relevant_no_males,Percentage_non_male,char_pct_num,Criteria,Director,Total_team,female_team,Team_percentage,team_pct_num,Metacritic_Kaggle,Metacritic_RAWG,Rating_RAWG_Audience,RatingsCount_RAWG,Destructoid,IGN,GameSpot,Avg_Reviews
0,JRN,Journey,Mar-12,2012,Adventure,Adventure,NaN,Thatgamecompany,Sony Computer Entertainment,USA,PS3,12,Non-Binary,1,1,0,4,100%,100.0,TR,M,7,1,14%,14.0,9.2,92,4.32,2470,9.0,9.0,9.5,9.2
1,TWD,The Walking Dead,Apr-12,2012,Adventure,Episodic adventure,The Walking Dead,Telltale Games,Telltale Games,USA,Multi,18,No,1,0,14,9,39%,39.0,SR,M,16,0,0%,0.0,8.5,92,4.36,2498,8.6,9.3,8.5,8.7
2,BL2,Borderlands 2,Sep-12,2012,Action,FPS,Borderlands,Gearbox Software,2K Games,USA,Multi,18,No,4,1,8,4,33%,33.0,MS,M,7,0,0%,0.0,8.1,89,4.01,3329,8.5,9.0,8.5,8.5
3,HLM,Hotline Miami,Oct-12,2012,Action,Top-down Shooter,NaN,Dennaton Games,Devolver Digital,SWE,Multi,18,No,2,0,8,2,20%,20.0,TR,M,2,0,0%,0.0,8.6,85,4.38,2664,9.0,8.8,8.5,8.7
4,BSI,BioShock Infinite,Mar-13,2013,Action,FPS,BioShock,Irrational Games,2K Games,USA,Multi,18,No,1,0,3,4,57%,57.0,TR,M,4,0,0%,0.0,9.4,94,4.38,4187,10.0,9.5,9.0,9.5


### 5.4 Building `characters.csv` (character level)

Joins `df_characters_aaa` with the four sexualization dimensions and the `Total` index from `df_sexualization`. The `Game` column is kept as the foreign key to `Game_Id` in `games.csv`.

In [32]:
characters = df_characters_aaa.merge(
    df_sexualization[["Id", "Sexualized_clothing", "Trophy",
                      "Damsel in Distress", "Sexualized Cutscenes", "Total"]],
    on="Id",
    how="left"
)

print(f"characters.csv: {characters.shape[0]} rows x {characters.shape[1]} columns")
characters.head()

characters.csv: 578 rows x 17 columns


,Name,Gender,Game,Age,Age_range,Playable,Sexualization,Id,Species,Side,Relevance,Romantic_Interest,Sexualized_clothing,Trophy,Damsel in Distress,Sexualized Cutscenes,Total
0,Farah,Female,CODMW,27,Adult,1,0,CODMW_Farah,Human,P,PA,No,0,0,0,0,0
1,Protagonist,Custom,PSS,Teenager,Teenager,1,0,PSS_Protagonist,Human,P,PA,No,0,0,0,0,0
2,Magnolia,Female,PSS,Elderly,Elderly,0,0,PSS_Magnolia,Human,P,SC,No,0,0,0,0,0
3,Sonia,Female,PSS,26,Adult,0,0,PSS_Sonia,Human,P,SC,No,0,0,0,0,0
4,Marnie,Female,PSS,Teenager,Teenager,0,0,PSS_Marnie,Human,B,MC,No,0,0,0,0,0


### 5.5 Consistency check before export

The final sample must contain 60 games and 578 characters, and all release years must fall within 2012-2022, the coverage of the Kaggle dataset.

In [33]:
print(f"Characters (characters.csv): {len(characters)}")
print(f"Games (games.csv): {len(games)}")
print("\nYears present in the final sample:")
print(sorted(games["year"].dropna().astype(int).unique()))

Characters (characters.csv): 578
Games (games.csv): 60

Years present in the final sample:
[2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]


### 5.6 Export

In [34]:
os.makedirs(PROCESSED_DIR, exist_ok=True)
games.to_csv(os.path.join(PROCESSED_DIR, "games.csv"), index=False)
characters.to_csv(os.path.join(PROCESSED_DIR, "characters.csv"), index=False)

print("Exported: data/processed/games.csv and data/processed/characters.csv")

Exported: data/processed/games.csv and data/processed/characters.csv
